# Connectome — sprint 18: **los árboles, con la cuenta hecha**

**Inputs:** train part1/part2, valid. **CPU** (no hace falta GPU: Session options → Accelerator → None). **Save & Run All.** ~2,5 h.

---

## Por qué volvemos a esto

En septiembre probamos LightGBM (la misma familia que XGBoost) y lo descartamos por tiempo de inferencia. Nunca medimos lo único que importaba. Medido ahora, sobre las predicciones que habían quedado guardadas:

| | correlación con el promedio del ensamble |
|---|---|
| otra LSTM como las nuestras | 0.985 |
| la convolucional del sprint 16 | 0.978 |
| **los árboles** | **0.918** |

Son, por lejos, lo más distinto que conseguimos en todo el proyecto.

## La cuenta

Para un ensamble con puntaje **S** y un modelo nuevo con puntaje **s** y correlación **r** con ese ensamble, mezclarlos mejora **si y solo si  s > S × r**. Con S = **0.6755** (nuestras 4 redes medidas en las mismas 200 secuencias que usa este notebook):

| modelo nuevo | r | tiene que pasar de | dio | resultado |
|---|---|---|---|---|
| otra LSTM | 0.985 | 0.665 | ~0.684 | casi igualar al ensamble entero: imposible en la práctica |
| la convolucional | 0.978 | 0.661 | 0.654 | **0.0000**, le faltaron 0.007 |
| **los árboles de septiembre** | **0.918** | **0.620** | 0.625 | **+0.0001**: justo en el límite |

Y como su correlación es baja, la pendiente ahí es empinada:

| si los árboles dieran | el ensamble ganaría |
|---|---|
| 0.640 | +0.0019 |
| 0.650 | **+0.0042** |
| 0.660 | **+0.0075** |
| 0.670 | **+0.0117** |

**No hace falta que los árboles se acerquen a las redes.** Con que suban 0.025 (de 0.625 a 0.650) es la mejora más grande en un mes.

## Qué cambia respecto de septiembre

1. **3,3 veces más datos.** Septiembre usó 300 filas por secuencia = 3,2M filas, el **1,5 %** de train. Acá van 1.000 por secuencia = **10,6M filas**. El límite es la memoria, no el tiempo.
2. **Las 40 features de memoria larga**: desvíos respecto de promedios móviles de 100, 1.000 y 10.000 filas, más volatilidad lenta. **Un árbol no tiene estado**, no puede calcularse esto solo; nuestras redes lo consiguen gratis por ser recurrentes y es su ventaja principal. En la prueba chica de la laptop, cuatro de estas features aparecieron entre las más importantes de entrada.
3. **Sacar 410 columnas muertas.** De las 560 de septiembre, las 150 mejores tienen el **90,3 %** de la ganancia. Sacarlas es lo que libera memoria para el punto 1.

## Lo que hay que mirar al final

El número clave es **el puntaje de los árboles solos**, contra el **0.6253** de septiembre:

- **≥ 0.650** → entra al ensamble y es la mejora grande que buscamos
- 0.635–0.650 → aporta poco pero real; hay que ver si paga el tiempo de inferencia
- ≤ 0.630 → los árboles quedan cerrados con evidencia

El tiempo de inferencia es el otro problema y está medido aparte: 600 árboles cuestan **58 µs por fila por target**, y nos quedan 30 µs libres. La salida: correrlos solo en las filas que la métrica puntúa. Con el modelo de la máscara, corriéndolos en el 25 % de las filas se atrapa el **68,7 %** del peso de la métrica y entran **623 árboles**. Eso se resuelve después de saber si los árboles mejoraron.

In [ ]:
import os, sys, json, numpy as np
print('python', sys.version.split()[0])
import lightgbm as lgb; print('lightgbm', lgb.__version__)
for p in ['/kaggle/input/datasets/joaquinfiorina/connectome-train-part1/train_part1.parquet', '/kaggle/input/datasets/joaquinfiorina/connectome-train-part2/train_part2.parquet'] + ['/kaggle/input/datasets/joaquinfiorina/connectome-valid/valid.parquet']:
    assert os.path.isfile(p), f'Falta input: {p}'
import psutil; print('RAM disponible: %.1f GB' % (psutil.virtual_memory().total/1e9))
assert psutil.virtual_memory().total > 20e9, 'Hace falta la sesion de CPU (30 GB), no la de GPU (13 GB)'
os.makedirs('/kaggle/working/results', exist_ok=True); print('Inputs OK')


In [ ]:
%%writefile utils.py
"""Competition callback and clipped block-wise Weighted Pearson scorer."""
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import numpy as np
import pyarrow.parquet as pq

SEQUENCE_LENGTH = 20_000
WARMUP = 99
N_FEATURES = 112
FEATURE_COLUMNS = [f'i{i}_{g}{j}' for i in range(2)
                   for g, n in [('p', 22), ('v', 22), ('dp', 4), ('dv', 4)] for j in range(n)]
FEATURE_COLUMNS += [f'a{i}' for i in range(8)]
TARGET_COLUMNS = ('t0', 't1')
METRIC_CLIP = 2.0


@dataclass(frozen=True)
class DataPoint:
    seq_ix: int
    step_in_seq: int
    need_prediction: bool
    state: np.ndarray


def weighted_pearson(target, prediction):
    target = np.asarray(target, dtype=np.float32)
    prediction = np.asarray(prediction, dtype=np.float32)
    if target.shape != prediction.shape or target.ndim != 1:
        raise ValueError('expected equal one-dimensional target and prediction arrays')
    if not np.isfinite(target).all() or not np.isfinite(prediction).all():
        raise ValueError('target/prediction contains nonfinite values')
    # The metric clips both arrays. Stored targets and submitted predictions
    # remain untouched outside this calculation.
    target = np.clip(target, -METRIC_CLIP, METRIC_CLIP).astype(np.float64)
    prediction = np.clip(prediction, -METRIC_CLIP, METRIC_CLIP).astype(np.float64)
    weights = np.abs(target)
    total = weights.sum()
    if total < 1e-8:
        return 0.0
    target_mean = np.sum(weights * target) / total
    prediction_mean = np.sum(weights * prediction) / total
    target_centered = target - target_mean
    prediction_centered = prediction - prediction_mean
    covariance = np.sum(weights * target_centered * prediction_centered) / total
    target_std = np.sqrt(np.sum(weights * target_centered**2) / total)
    prediction_std = np.sqrt(np.sum(weights * prediction_centered**2) / total)
    if target_std <= 1e-8 or prediction_std <= 1e-8:
        return 0.0
    return float(np.clip(covariance / (target_std * prediction_std), -1.0, 1.0))


class BlockAccumulator:
    def __init__(self):
        self.scores = []
        self.blocks_seen = 0
        self.selected_rows = 0
        self.effective_rows = 0

    def add(self, targets, predictions, mask):
        targets = np.asarray(targets, dtype=np.float32)
        predictions = np.asarray(predictions, dtype=np.float32)
        mask = np.asarray(mask, dtype=bool)
        if targets.shape != (SEQUENCE_LENGTH, 2) or predictions.shape != targets.shape or mask.shape != (SEQUENCE_LENGTH,):
            raise ValueError('one accumulator call must contain exactly one complete 20k block')
        if np.any(mask[:WARMUP]):
            raise ValueError('warm-up rows cannot be scored')
        if not np.isfinite(targets).all() or not np.isfinite(predictions[WARMUP:]).all():
            raise ValueError('nonfinite labels or required predictions')
        selected_targets = targets[mask]
        selected_predictions = predictions[mask]
        eligible = all(np.any(selected_targets[:, j] > 0) and
                       np.any(selected_targets[:, j] < 0) for j in range(2))
        self.blocks_seen += 1
        self.selected_rows += int(mask.sum())
        if eligible:
            values = [weighted_pearson(selected_targets[:, j], selected_predictions[:, j])
                      for j in range(2)]
            self.scores.append(values)
            self.effective_rows += int(mask.sum())

    def result(self):
        if not self.scores:
            raise ValueError('no eligible blocks')
        per_side = np.mean(np.asarray(self.scores, dtype=np.float64), axis=0)
        return {'t0': float(per_side[0]), 't1': float(per_side[1]),
                'weighted_pearson': float(per_side.mean()), 'blocks': self.blocks_seen,
                'eligible_blocks': len(self.scores), 'excluded_blocks': self.blocks_seen-len(self.scores),
                'selected_rows': self.selected_rows, 'effective_scored_rows': self.effective_rows,
                'metric_clip': [-METRIC_CLIP, METRIC_CLIP],
                'aggregation': 'equal mean of eligible block-wise two-target clipped Weighted Pearson'}


def validate_sequence(table):
    if table.num_rows != SEQUENCE_LENGTH:
        raise ValueError('each row group must be exactly one 20k sequence')
    ids = table['seq_ix'].to_numpy()
    if not np.all(ids == ids[0]):
        raise ValueError('multiple sequence IDs in one row group')
    if not np.array_equal(table['step_in_seq'].to_numpy(), np.arange(SEQUENCE_LENGTH)):
        raise ValueError('incorrect step order')
    need = table['need_prediction'].to_numpy().astype(bool)
    if not np.array_equal(need, np.arange(SEQUENCE_LENGTH) >= WARMUP):
        raise ValueError('incorrect warm-up contract')
    return int(ids[0]), need


class ScorerStepByStep:
    def __init__(self, dataset_path: str | Path):
        self.parquet = pq.ParquetFile(dataset_path)
        expected = ['seq_ix', 'step_in_seq', 'need_prediction', 'is_scored', *FEATURE_COLUMNS, *TARGET_COLUMNS]
        if self.parquet.schema_arrow.names != expected:
            raise ValueError('local scoring requires the labeled validation schema')

    def score(self, model):
        accumulator = BlockAccumulator()
        seen = set()
        for group in range(self.parquet.num_row_groups):
            table = self.parquet.read_row_group(group, use_threads=False)
            seq, need = validate_sequence(table)
            if seq in seen:
                raise ValueError('duplicate sequence ID')
            seen.add(seq)
            features = np.column_stack([table[c].to_numpy() for c in FEATURE_COLUMNS]).astype(np.float32)
            targets = np.column_stack([table[c].to_numpy() for c in TARGET_COLUMNS]).astype(np.float32)
            predictions = np.full((SEQUENCE_LENGTH, 2), np.nan, dtype=np.float32)
            for step in range(SEQUENCE_LENGTH):
                point = DataPoint(seq, step, bool(need[step]), features[step])
                value = model.predict(point)
                if not need[step]:
                    if value is not None:
                        raise ValueError('return None during warm-up AFTER updating state')
                else:
                    value = np.asarray(value, dtype=np.float32)
                    if value.shape != (2,) or not np.isfinite(value).all():
                        raise ValueError('return two finite scores on every required row')
                    predictions[step] = value
            accumulator.add(targets, predictions, table['is_scored'].to_numpy().astype(bool) & need)
        result = accumulator.result()
        result['rows_seen'] = len(seen)*SEQUENCE_LENGTH
        result['predictions_required'] = len(seen)*(SEQUENCE_LENGTH-WARMUP)
        return result


In [ ]:
%%writefile train_lgbm_deltas_v1.py
r"""LightGBM sobre features causales (112 crudas + deltas a varios lags), un modelo por target.

Pensado como SEGUNDO modelo para ensamblar con la GRU: aprende cosas distintas (umbrales,
interacciones entre pocas columnas) y en inferencia cuesta ~5-10 us/fila.

* Entrena con una muestra de filas (cada fila es un ejemplo independiente): N secuencias de
  train x M filas al azar por secuencia (solo filas con need_prediction).
* Loss alineada a la métrica: target recortado a [-2, 2] y peso |target|.
* Evalúa con la métrica oficial (utils.BlockAccumulator) en los mismos grupos de valid que
  EXPERIMENTS.md (--valid-groups 10 --seed 2026 -> baseline 0.568).
* Guarda los modelos (.txt), la spec de features, métricas y las predicciones de valid (para
  ensamblar offline con la GRU). Si se pasa --gru-model best_model.pt, evalúa también la mezcla.

Uso local (desde challenge/):
    python model/lgbm_deltas/train_lgbm_deltas_v1.py --train-groups 300 --rows-per-group 300
Uso Kaggle (CPU, sin GPU):
    python train_lgbm_deltas_v1.py --train-parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part1/train_part1.parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part2/train_part2.parquet --valid-parquet /kaggle/input/datasets/joaquinfiorina/connectome-valid/valid.parquet --train-groups 2000 --rows-per-group 500 --output-dir /kaggle/working/results --run-name lgbm_deltas_v1
"""
from __future__ import annotations

import argparse
import csv
import json
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pyarrow.parquet as pq

HERE = Path(__file__).resolve().parent
ROOT = HERE.parents[2] if len(HERE.parents) > 2 else HERE
for p in (ROOT, HERE):
    if str(p) not in sys.path:
        sys.path.insert(0, str(p))
from utils import (BlockAccumulator, FEATURE_COLUMNS, SEQUENCE_LENGTH, TARGET_COLUMNS,  # noqa: E402
                   WARMUP, validate_sequence, weighted_pearson)


# ----------------------------------------------------------------------------- datos
class ParquetGroups:
    def __init__(self, paths):
        self.files = [pq.ParquetFile(p) for p in paths]
        self.index = [(f, g) for f in self.files for g in range(f.num_row_groups)]
        self.has_scored = all("is_scored" in f.schema_arrow.names for f in self.files)

    def __len__(self):
        return len(self.index)

    def read(self, i):
        f, g = self.index[i]
        cols = ["seq_ix", "step_in_seq", "need_prediction", *FEATURE_COLUMNS, *TARGET_COLUMNS]
        if self.has_scored:
            cols.append("is_scored")
        t = f.read_row_group(g, columns=cols, use_threads=False)
        seq_ix, need = validate_sequence(t)
        x = np.column_stack([t[c].to_numpy() for c in FEATURE_COLUMNS]).astype(np.float32)
        y = np.column_stack([t[c].to_numpy() for c in TARGET_COLUMNS]).astype(np.float32)
        x[~np.isfinite(x)] = 0.0
        scored = t["is_scored"].to_numpy().astype(bool) if self.has_scored else None
        return seq_ix, need, x, y, scored


def select_groups(total, count, seed):
    if not 1 <= count <= total:
        raise ValueError(f"count debe estar entre 1 y {total}")
    return sorted(np.random.default_rng(seed).choice(total, count, replace=False).tolist())



# ----------------------------------------------------------------------------- métrica GLOBAL (nueva, 2026-09)
class GlobalWP:
    """Weighted Pearson global: junta TODAS las filas scoreadas de todas las secuencias y calcula
    una sola correlación por target (clip ±2, peso |y|), luego promedia t0 y t1.
    Reemplaza a la métrica por secuencia (utils.BlockAccumulator), que quedó obsoleta."""

    def __init__(self):
        self.s = np.zeros((2, 6), dtype=np.float64)   # por target: sum w, wy, wp, wyy, wpp, wyp

    def add(self, targets, predictions, mask):
        y = np.clip(targets[mask].astype(np.float64), -2.0, 2.0)
        p = np.clip(predictions[mask].astype(np.float64), -2.0, 2.0)
        w = np.abs(y)
        for j in range(2):
            wj, yj, pj = w[:, j], y[:, j], p[:, j]
            self.s[j] += [wj.sum(), (wj * yj).sum(), (wj * pj).sum(),
                          (wj * yj * yj).sum(), (wj * pj * pj).sum(), (wj * yj * pj).sum()]

    def result(self):
        out = {}
        for j, name in enumerate(TARGET_COLUMNS):
            W, wy, wp, wyy, wpp, wyp = self.s[j]
            if W < 1e-8:
                out[name] = 0.0
                continue
            my, mp = wy / W, wp / W
            cov = wyp / W - my * mp
            vy, vp = wyy / W - my * my, wpp / W - mp * mp
            out[name] = float(np.clip(cov / np.sqrt(vy * vp), -1.0, 1.0)) if vy > 1e-16 and vp > 1e-16 else 0.0
        out["weighted_pearson"] = (out["t0"] + out["t1"]) / 2
        return out


# ----------------------------------------------------------------------------- features
# --- features de memoria larga (mismas que --ema-features del entrenador de redes)
_IX = {c: i for i, c in enumerate(FEATURE_COLUMNS)}
EMA_COLS = ["i0_p0", "i0_p11", "i0_v0", "i0_v11", "i1_p0", "i1_p11", "i1_v0", "i1_v11", "a0", "a1", "a2", "a4"]
EMA_IX = [_IX[c] for c in EMA_COLS]
EMA_SPANS = (100, 1000, 10000)
EMA_VOL_COLS = ["i0_p0", "i0_p11", "i0_v0", "i1_p0"]
EMA_NAMES = ([f"{c}_dif_ema{s}" for s in EMA_SPANS for c in EMA_COLS] +
             [f"{c}_vol_lenta" for c in EMA_VOL_COLS])
try:
    from scipy.signal import lfilter as _lfilter
except Exception:
    _lfilter = None


def _ema(x, span):
    """Promedio móvil exponencial causal por columna. ema[0] = x[0]."""
    a = 2.0 / (span + 1.0)
    if _lfilter is not None:
        zi = (x[0] * (1 - a))[None, :]
        return _lfilter([a], [1.0, -(1 - a)], x, axis=0, zi=zi)[0]
    out = np.empty_like(x)
    m = x[0].copy()
    for t in range(len(x)):
        m += a * (x[t] - m)
        out[t] = m
    return out


def ema_features(x):
    """(T, 112) -> (T, 40): desvío respecto de promedios móviles largos + volatilidad lenta."""
    sub = x[:, EMA_IX].astype(np.float32)
    parts, e100 = [], None
    for span in EMA_SPANS:
        e = _ema(sub, span).astype(np.float32)
        if span == EMA_SPANS[0]:
            e100 = e
        parts.append(sub - e)
    vi = [EMA_COLS.index(c) for c in EMA_VOL_COLS]
    parts.append(_ema(np.abs(sub[:, vi] - e100[:, vi]), 1000).astype(np.float32))
    return np.clip(np.concatenate(parts, axis=1), -10.0, 10.0).astype(np.float32)


def feature_names(lags, ema=False):
    names = list(FEATURE_COLUMNS)
    for k in lags:
        names += [f"{c}_delta_{k}" for c in FEATURE_COLUMNS]
    if ema:
        names += EMA_NAMES
    return names


def build_features(x, lags, ema=False, keep=None):
    """x: (T, 112) de UNA secuencia. Causal: solo pasado.
    Las primeras k filas de cada delta quedan NaN (LightGBM las maneja nativamente).
    `keep`: índices de columnas a conservar (para no arrastrar las features que no se usan)."""
    out = [x]
    for k in lags:
        d = np.full_like(x, np.nan)
        d[k:] = x[k:] - x[:-k]
        out.append(d)
    if ema:
        out.append(ema_features(x))
    f = np.concatenate(out, axis=1)
    return f if keep is None else f[:, keep]


# ----------------------------------------------------------------------------- GRU opcional (para evaluar la mezcla)
def gru_predict_factory(path):
    import torch
    from torch import nn
    ck = torch.load(path, map_location="cpu", weights_only=False)
    sd, cfg, pipe = ck["state_dict"], ck["config"], ck["pipeline"]
    if pipe.get("top_k", 0) or pipe.get("sentinel_flags", False):
        raise ValueError("la mezcla solo soporta GRU con 112 entradas crudas")
    core = nn.GRU if cfg["cell"] == "gru" else nn.LSTM
    model = nn.ModuleDict({"core": core(ck["n_inputs"], cfg["hidden"], num_layers=cfg["layers"], batch_first=True),
                           "head": nn.Linear(cfg["hidden"], 2)})
    model.load_state_dict(sd)
    model.eval()
    mean, std = pipe["mean"], pipe["std"]

    @torch.no_grad()
    def predict(x):
        z = torch.from_numpy(((x - mean) / std).astype(np.float32)).unsqueeze(0)
        out, _ = model["core"](z)
        return model["head"](out).squeeze(0).numpy()
    return predict


# ----------------------------------------------------------------------------- main
def main():
    ap = argparse.ArgumentParser(description=__doc__, formatter_class=argparse.RawDescriptionHelpFormatter)
    ap.add_argument("--train-parquet", nargs="+", default=[str(ROOT / "datasets" / "train.parquet")])
    ap.add_argument("--valid-parquet", nargs="+", default=[str(ROOT / "datasets" / "valid.parquet")])
    ap.add_argument("--train-groups", default="all", help="N o 'all'")
    ap.add_argument("--rows-per-group", type=int, default=300)
    ap.add_argument("--valid-groups", type=int, default=10)
    ap.add_argument("--seed", type=int, default=2026)
    ap.add_argument("--lags", type=int, nargs="+", default=[1, 5, 20, 100])
    ap.add_argument("--n-estimators", type=int, default=600)
    ap.add_argument("--learning-rate", type=float, default=0.05)
    ap.add_argument("--num-leaves", type=int, default=63)
    ap.add_argument("--min-data-in-leaf", type=int, default=200)
    ap.add_argument("--feature-fraction", type=float, default=0.5)
    ap.add_argument("--bagging-fraction", type=float, default=0.8)
    ap.add_argument("--lambda-l2", type=float, default=1.0)
    ap.add_argument("--threads", type=int, default=-1)
    ap.add_argument("--ema-features", action="store_true", help="agregar las 40 features de memoria larga "
                    "(promedios móviles de 100/1.000/10.000 filas). Es lo que un árbol no puede calcularse solo")
    ap.add_argument("--top-features", type=int, default=0, help="quedarse con las N columnas de mayor ganancia "
                    "según --importance (0 = todas). Las features de memoria larga se conservan siempre")
    ap.add_argument("--importance", default=None, help="feature_importance.json de una corrida anterior")
    ap.add_argument("--ens-preds", nargs="*", default=[], help="valid_preds_*.npz del ensamble de redes: "
                    "reporta correlación y ganancia honesta (peso elegido en media valid, medido en la otra media)")
    ap.add_argument("--gru-model", default=None, help="best_model.pt de train_official_gpu_v1 para evaluar la mezcla")
    ap.add_argument("--output-dir", default=str(HERE / "results"))
    ap.add_argument("--run-name", default=None)
    args = ap.parse_args()

    import lightgbm as lgb

    run_dir = Path(args.output_dir) / (args.run_name or datetime.now().strftime("run_%Y%m%d_%H%M%S"))
    run_dir.mkdir(parents=True, exist_ok=False)
    names_all = feature_names(args.lags, args.ema_features)
    keep = None
    if args.top_features and args.importance:
        imp = dict(json.loads(Path(args.importance).read_text(encoding="utf-8")))
        base = [n for n in names_all if n not in EMA_NAMES]
        orden = sorted(base, key=lambda n: -imp.get(n, 0.0))[:args.top_features]
        quedan = set(orden) | (set(EMA_NAMES) if args.ema_features else set())
        keep = np.array([i for i, n in enumerate(names_all) if n in quedan])
        names = [names_all[i] for i in keep]
        print(f"columnas: {len(names_all)} -> {len(names)} "
              f"(top {args.top_features} por ganancia"
              f"{' + las ' + str(len(EMA_NAMES)) + ' de memoria larga' if args.ema_features else ''})", flush=True)
    else:
        names = names_all
    train = ParquetGroups(args.train_parquet)
    valid = ParquetGroups(args.valid_parquet)
    train_ids = list(range(len(train))) if str(args.train_groups) == "all" else select_groups(len(train), int(args.train_groups), args.seed)
    valid_ids = select_groups(len(valid), args.valid_groups, args.seed + 1)
    cfg = {**vars(args), "train_ids": train_ids, "valid_ids": valid_ids, "n_features": len(names)}
    (run_dir / "run_config.json").write_text(json.dumps(cfg, indent=2), encoding="utf-8")
    print(f"features={len(names)} (112 crudas + deltas {args.lags}"
          f"{' + 40 de memoria larga' if args.ema_features else ''})  "
          f"train seqs={len(train_ids)} x {args.rows_per_group} filas "
          f"= {len(train_ids)*args.rows_per_group/1e6:.1f}M filas, "
          f"~{len(train_ids)*args.rows_per_group*len(names)*4/1e9:.1f} GB", flush=True)

    # ---- muestra de train
    t0 = time.time()
    rng = np.random.default_rng(args.seed + 2)
    xs, ys = [], []
    for n, i in enumerate(train_ids, 1):
        _, need, x, y, _ = train.read(i)
        f = build_features(x, args.lags, args.ema_features, keep)
        rows = np.flatnonzero(need)
        take = rng.choice(rows, min(args.rows_per_group, len(rows)), replace=False)
        xs.append(f[take])
        ys.append(y[take])
        if n % 200 == 0 or n == len(train_ids):
            print(f"  muestra: {n}/{len(train_ids)} seqs  ({time.time()-t0:.0f}s)", flush=True)
    X = np.concatenate(xs)
    Y = np.concatenate(ys)
    del xs, ys
    print(f"X: {X.shape}  ({X.nbytes/1e9:.2f} GB)", flush=True)

    # ---- entrenar un modelo por target (loss alineada: clip +-2, peso |y|)
    params = dict(objective="regression", learning_rate=args.learning_rate, num_leaves=args.num_leaves,
                  min_data_in_leaf=args.min_data_in_leaf, feature_fraction=args.feature_fraction,
                  bagging_fraction=args.bagging_fraction, bagging_freq=1, lambda_l2=args.lambda_l2,
                  num_threads=args.threads, verbose=-1, seed=args.seed)
    models = []
    y0 = np.clip(Y[:, 0], -2.0, 2.0)
    ds = lgb.Dataset(X, label=y0, weight=np.abs(y0), feature_name=names, free_raw_data=True)
    ds.construct()
    del X   # el Dataset ya tiene los datos binneados; liberar los 4 bytes/valor crudos
    for j, tname in enumerate(TARGET_COLUMNS):
        yj = np.clip(Y[:, j], -2.0, 2.0)
        w = np.abs(yj)
        ds.set_label(yj)
        ds.set_weight(w)
        t = time.time()
        booster = lgb.train(params, ds, num_boost_round=args.n_estimators)
        booster.save_model(str(run_dir / f"lgbm_{tname}.txt"))
        models.append(booster)
        print(f"  {tname}: {args.n_estimators} árboles en {time.time()-t:.0f}s", flush=True)
    del Y

    # ---- importancia (top 25)
    imp = {n: 0.0 for n in names}
    for b in models:
        for n, g in zip(names, b.feature_importance("gain")):
            imp[n] += float(g)
    top = sorted(imp.items(), key=lambda kv: -kv[1])[:25]
    (run_dir / "feature_importance.json").write_text(json.dumps(sorted(imp.items(), key=lambda kv: -kv[1]), indent=1), encoding="utf-8")
    print("top 25 features (ganancia):", flush=True)
    for n, g in top:
        print(f"   {n:22s} {g:12.0f}", flush=True)

    # ---- evaluación oficial en valid
    gru_predict = gru_predict_factory(args.gru_model) if args.gru_model else None
    acc, gacc = BlockAccumulator(), GlobalWP()
    mix_w = (0.2, 0.3, 0.4, 0.5)
    accs_mix = {a: BlockAccumulator() for a in mix_w} if gru_predict else {}
    gaccs_mix = {a: GlobalWP() for a in mix_w} if gru_predict else {}
    acc_gru, gacc_gru = (BlockAccumulator(), GlobalWP()) if gru_predict else (None, None)
    # predicciones del ensamble de redes, para medir correlación y ganancia honesta
    ens_src = []
    for ruta in args.ens_preds:
        d = np.load(ruta)
        ens_src.append(({int(v): i for i, v in enumerate(d["seq_ix"])}, d["preds"]))
    if ens_src:
        print(f"ensamble de referencia: {len(ens_src)} modelos", flush=True)
    ens_rows = []    # (y, arboles, ensamble, seq) solo de las filas puntuadas

    rows_out, preds_store = [], {}
    t = time.time()
    for i in valid_ids:
        seq_ix, need, x, y, scored = valid.read(i)
        f = build_features(x, args.lags, args.ema_features, keep)
        p = np.column_stack([b.predict(f, num_threads=args.threads) for b in models]).astype(np.float32)
        mask = need & scored
        acc.add(y, p, mask)
        gacc.add(y, p, mask)
        yy, pp = y[mask], p[mask]
        elig = all(np.any(yy[:, j] > 0) and np.any(yy[:, j] < 0) for j in range(2))
        row = {"seq_ix": seq_ix}
        if elig:
            row.update(wp_t0=weighted_pearson(yy[:, 0], pp[:, 0]), wp_t1=weighted_pearson(yy[:, 1], pp[:, 1]))
            row["wp"] = (row["wp_t0"] + row["wp_t1"]) / 2
        preds_store[str(seq_ix)] = p
        if ens_src and all(int(seq_ix) in pos for pos, _ in ens_src):
            e = np.mean([pr[pos[int(seq_ix)]] for pos, pr in ens_src], 0).astype(np.float32)
            ens_rows.append((y[mask], p[mask], e[mask], np.full(int(mask.sum()), int(seq_ix))))
        if gru_predict:
            g = gru_predict(x).astype(np.float32)
            acc_gru.add(y, g, mask)
            gacc_gru.add(y, g, mask)
            for a in mix_w:                        # a = peso de LightGBM
                m = (a * p + (1 - a) * g).astype(np.float32)
                accs_mix[a].add(y, m, mask)
                gaccs_mix[a].add(y, m, mask)
        rows_out.append(row)
    res, gres = acc.result(), gacc.result()
    print(f"\nLightGBM  GLOBAL WP={gres['weighted_pearson']:.4f} (t0={gres['t0']:.4f}, t1={gres['t1']:.4f})   "
          f"[por secuencia (métrica vieja): {res['weighted_pearson']:.4f}]  ({time.time()-t:.0f}s)", flush=True)
    out = {"lgbm_global": gres, "lgbm_per_sequence": res, "config": cfg}
    if gru_predict:
        rg, grg = acc_gru.result(), gacc_gru.result()
        print(f"GRU sola  GLOBAL WP={grg['weighted_pearson']:.4f} (t0={grg['t0']:.4f}, t1={grg['t1']:.4f})   [por secuencia: {rg['weighted_pearson']:.4f}]", flush=True)
        out["gru_global"], out["gru_per_sequence"], out["mix"] = grg, rg, {}
        for a in mix_w:
            rm, grm = accs_mix[a].result(), gaccs_mix[a].result()
            out["mix"][str(a)] = {"global": grm, "per_sequence": rm}
            print(f"mezcla {a:.1f}*LGBM + {1-a:.1f}*GRU  GLOBAL WP={grm['weighted_pearson']:.4f} (t0={grm['t0']:.4f}, t1={grm['t1']:.4f})   [por secuencia: {rm['weighted_pearson']:.4f}]", flush=True)
    if ens_rows:
        YY = np.clip(np.concatenate([r[0] for r in ens_rows]), -2, 2).astype(np.float64)
        PA = np.clip(np.concatenate([r[1] for r in ens_rows]), -2, 2).astype(np.float64)
        PE = np.clip(np.concatenate([r[2] for r in ens_rows]), -2, 2).astype(np.float64)
        SQ = np.concatenate([r[3] for r in ens_rows])
        WW = np.abs(YY)

        def _wp(pred, m):
            vals = []
            for j in (0, 1):
                w, y, q = WW[m, j], YY[m, j], pred[m, j]
                sw = w.sum()
                my, mq = (w * y).sum() / sw, (w * q).sum() / sw
                cov = (w * (y - my) * (q - mq)).sum() / sw
                vy, vq = (w * (y - my) ** 2).sum() / sw, (w * (q - mq) ** 2).sum() / sw
                vals.append(cov / np.sqrt(vy * vq))
            return (vals[0] + vals[1]) / 2

        A = (SQ % 2 == 0)
        todo = np.ones(len(YY), bool)
        r_corr = float(np.corrcoef(PA[:, 0], PE[:, 0])[0, 1])
        s_arb, s_ens = _wp(PA, todo), _wp(PE, todo)
        print(f"\n{'='*64}")
        print(f"ARBOLES vs ENSAMBLE DE REDES   ({len(YY):,} filas puntuadas, {len(np.unique(SQ))} secuencias)")
        print(f"{'='*64}")
        print(f"  arboles solos          {s_arb:.4f}")
        print(f"  ensamble de redes      {s_ens:.4f}")
        print(f"  correlacion entre los dos: {r_corr:.3f}")
        print(f"  umbral para aportar: los arboles necesitan pasar de {s_ens:.4f} x {r_corr:.3f} = {s_ens*r_corr:.4f}"
              f"  ->  {'LO PASAN' if s_arb > s_ens*r_corr else 'NO lo pasan'} ({s_arb - s_ens*r_corr:+.4f})")
        pesos = np.linspace(0, 0.6, 61)
        gA = [_wp((1 - a) * PE + a * PA, A) for a in pesos]
        mejor = pesos[int(np.argmax(gA))]
        bB, mB = _wp(PE, ~A), _wp((1 - mejor) * PE + mejor * PA, ~A)
        print(f"\n  peso elegido en la mitad A: {mejor:.2f}")
        print(f"  medido en la mitad B:   solo redes {bB:.4f}  ->  con arboles {mB:.4f}   "
              f"GANANCIA HONESTA {mB - bB:+.4f}")
        print(f"{'='*64}\n", flush=True)
        out["vs_ensamble"] = {"arboles": s_arb, "ensamble": s_ens, "correlacion": r_corr,
                              "umbral": s_ens * r_corr, "peso_mitad_A": float(mejor),
                              "mitad_B_solo_redes": bB, "mitad_B_con_arboles": mB, "ganancia": mB - bB}
    (run_dir / "metrics.json").write_text(json.dumps(out, indent=2, default=str), encoding="utf-8")
    with (run_dir / "by_rowgroup.csv").open("w", newline="", encoding="utf-8") as fh:
        w = csv.DictWriter(fh, fieldnames=["seq_ix", "wp_t0", "wp_t1", "wp"])
        w.writeheader()
        w.writerows(rows_out)
    np.savez_compressed(run_dir / "valid_predictions.npz", **preds_store)
    (run_dir / "feature_spec.json").write_text(json.dumps(
        {"lags": args.lags, "names": names, "ema": bool(args.ema_features),
         "keep": None if keep is None else [int(v) for v in keep]}), encoding="utf-8")
    print(f"Resultados en {run_dir.resolve()}", flush=True)


if __name__ == "__main__":
    main()


## Importancia de las features de septiembre

Se usa para elegir las 150 columnas que valen la pena.

In [ ]:
import json
importancia = [["i0_p3", 1435958.9287567139], ["i0_p15", 897936.5677947998], ["i0_p0", 423059.29328918457], ["i0_p21", 421780.41205596924], ["i0_p14", 368988.7892665863], ["i0_v21", 299037.589384079], ["i0_v4", 279830.96279907227], ["i0_v0", 267610.81286239624], ["i0_dv2", 240219.86288452148], ["i0_p16", 238656.6718864441], ["i0_v15", 234266.5619354248], ["i0_v4_delta_5", 230083.71186828613], ["i0_dp1", 226019.8847618103], ["i0_dv1", 217854.37760543823], ["i0_p8", 205753.4681777954], ["i0_p19", 202928.77822494507], ["i0_p20", 196803.7650756836], ["i0_v4_delta_1", 190520.3189754486], ["i0_v19", 185371.59166908264], ["i0_v4_delta_20", 177941.73220443726], ["a6", 176518.38780212402], ["i0_v4_delta_100", 151493.2565059662], ["i0_v9", 146512.89982795715], ["i0_v20", 145800.30026435852], ["i0_dp0", 141931.6812019348], ["i0_p7", 130191.7539024353], ["i0_v15_delta_100", 128051.91555976868], ["i1_dv1", 125750.26552772522], ["a7", 124281.1964302063], ["i0_p2", 119668.37972831726], ["i0_p17", 117841.80644607544], ["i0_v2", 112743.48978996277], ["i0_v13", 111856.94515991211], ["i0_p1", 107458.5946598053], ["i0_v6", 105898.98683166504], ["i0_v3", 104592.24404335022], ["i0_p13", 100644.14566040039], ["i1_dp2", 98498.32820892334], ["i0_v5", 96330.19065475464], ["i0_dp2", 94411.5411453247], ["i0_v18", 94249.92359733582], ["i0_v13_delta_1", 91235.15048599243], ["i0_v17", 89391.84935951233], ["i0_v7", 82431.02255630493], ["i0_v16", 81745.72315597534], ["i0_v14", 80934.23603820801], ["a2", 79733.14362335205], ["i1_dp0", 76898.56738471985], ["a6_delta_5", 76297.97491073608], ["a0", 75309.44554138184], ["i0_v1", 73867.44896697998], ["i0_p12", 72313.72560501099], ["i0_v21_delta_5", 71827.06241989136], ["i0_v12", 71100.86951637268], ["i0_v8", 68545.41995620728], ["i0_v15_delta_20", 67444.3829498291], ["i0_v0_delta_20", 63433.62159729004], ["i1_dv2", 63129.56814575195], ["i0_v0_delta_100", 63088.90281677246], ["i0_v11", 62392.291149139404], ["a6_delta_1", 59574.30750656128], ["i0_p4", 59063.69980239868], ["i0_p9", 56782.61668395996], ["i0_v15_delta_5", 56160.80472564697], ["i0_v9_delta_100", 55034.6155128479], ["i1_v3", 54335.95909500122], ["i0_dp3", 53100.341762542725], ["i1_v11", 52276.16007041931], ["i1_v19", 52231.20003128052], ["i0_v21_delta_20", 51805.42687225342], ["i1_v17", 51710.180223464966], ["i0_p11", 51249.773151397705], ["i0_dv2_delta_1", 51004.2495098114], ["i0_v13_delta_5", 50816.43938064575], ["i1_v4", 50014.22857284546], ["i0_v5_delta_5", 49843.84107208252], ["a6_delta_20", 48634.054527282715], ["a3", 47425.270977020264], ["i1_v21", 47291.725116729736], ["i0_v19_delta_100", 46932.78887939453], ["i0_p5", 46325.408184051514], ["i1_v2", 45797.05274391174], ["i0_v10", 44860.51119041443], ["i0_p6", 44254.87943267822], ["i0_v21_delta_100", 42714.58175086975], ["a5", 42564.78944396973], ["i0_p10", 42563.580448150635], ["i1_v9", 42357.36979675293], ["i1_v0", 42200.506381988525], ["i1_v13", 41814.871046066284], ["i1_v1", 41714.6746635437], ["i1_v5", 41532.96731185913], ["i0_v9_delta_20", 41265.77491760254], ["i1_p5", 40696.26772689819], ["i1_v6", 40581.19643974304], ["i0_v0_delta_5", 39496.40062713623], ["i1_v7", 38831.54783821106], ["i0_v19_delta_5", 38078.034717559814], ["i1_dp1", 38015.19116401672], ["i0_v16_delta_1", 37927.65354537964], ["i1_v18", 37727.93092918396], ["i1_v12", 36618.02847290039], ["i1_v15", 36313.54641723633], ["i0_v9_delta_5", 35525.90330886841], ["i0_dv1_delta_1", 35135.82803726196], ["i1_v8", 34659.74631500244], ["i1_v10", 33957.28337478638], ["i1_v14", 33740.763175964355], ["i0_v5_delta_20", 32067.2879486084], ["i0_v19_delta_20", 31264.41843032837], ["i1_v20", 30326.937477111816], ["i1_v16", 30098.631065368652], ["i0_v20_delta_100", 28776.179889678955], ["i0_v20_delta_5", 28664.507747650146], ["a3_delta_5", 28162.31767654419], ["i0_v13_delta_20", 27918.057220458984], ["i0_p18", 26375.57306289673], ["i0_v17_delta_100", 25985.02000427246], ["i1_p13", 25943.91888999939], ["i0_v3_delta_1", 25700.997745513916], ["i0_v20_delta_20", 23588.033172607422], ["i0_v13_delta_100", 23421.95672607422], ["i0_dv3", 23266.393520355225], ["i0_v18_delta_5", 22654.65089416504], ["i0_v21_delta_1", 22288.269147872925], ["i0_v6_delta_100", 22068.742418289185], ["i0_v2_delta_100", 21528.516222000122], ["i0_v5_delta_1", 20011.025802612305], ["i0_v16_delta_5", 19916.967107772827], ["i0_v9_delta_1", 19868.800148010254], ["i0_p19_delta_1", 19147.85604095459], ["i0_v5_delta_100", 19096.80075454712], ["i1_dv2_delta_1", 18973.840881347656], ["i0_v6_delta_20", 18669.143508911133], ["i1_dv1_delta_1", 18485.892528533936], ["i0_dv1_delta_5", 18338.687740325928], ["i0_v19_delta_1", 18251.015031814575], ["i0_v0_delta_1", 16790.543348312378], ["i1_v4_delta_5", 16605.23997116089], ["a6_delta_100", 15392.31070137024], ["i0_dp2_delta_1", 14857.944526672363], ["a5_delta_5", 14715.343124389648], ["i0_v18_delta_1", 14687.767482757568], ["i0_v8_delta_5", 14662.860733032227], ["i0_dp0_delta_5", 14463.677547454834], ["i0_v6_delta_5", 14461.752914428711], ["i0_v17_delta_5", 14457.103969573975], ["i0_v20_delta_1", 14283.734481811523], ["i0_v18_delta_100", 14108.545124053955], ["i0_dv0", 13963.848976135254], ["i0_v17_delta_20", 13768.731178283691], ["i0_v10_delta_5", 13592.25549697876], ["i0_v3_delta_5", 13564.379495620728], ["i1_p17", 13488.428234100342], ["i1_v19_delta_5", 13279.192180633545], ["i0_v3_delta_100", 12891.722009658813], ["i1_v13_delta_1", 12868.621700286865], ["i0_dp0_delta_1", 12716.40400314331], ["i1_p16", 12423.19149017334], ["i1_dv0", 12311.614442825317], ["i0_v14_delta_5", 12260.35754776001], ["i1_v4_delta_1", 12103.129650115967], ["i0_dp2_delta_5", 12081.46244430542], ["i1_dv1_delta_5", 11965.107315063477], ["i0_v7_delta_100", 11522.552011489868], ["i0_dv0_delta_1", 11481.999996185303], ["i0_v2_delta_1", 11079.882081985474], ["i0_v16_delta_100", 10967.580709457397], ["i0_dv3_delta_5", 10835.16413116455], ["a4", 10786.443059921265], ["i0_v1_delta_100", 10686.967908859253], ["i1_v9_delta_5", 10672.9746799469], ["i0_v18_delta_20", 10342.72960281372], ["a5_delta_1", 10341.818508148193], ["i0_v1_delta_1", 10196.55164718628], ["i0_dp0_delta_20", 10195.72607421875], ["i0_dv1_delta_100", 10016.772987365723], ["i0_p14_delta_1", 9912.868404388428], ["i1_dp0_delta_5", 9666.55768585205], ["i1_v4_delta_100", 9323.919624328613], ["i0_dv1_delta_20", 9304.386737823486], ["a3_delta_20", 9259.004810333252], ["i0_v15_delta_1", 9050.479982376099], ["i0_v8_delta_100", 8978.548191070557], ["i1_v15_delta_20", 8955.735185623169], ["i1_v4_delta_20", 8925.776237487793], ["i1_dv3", 8848.562110900879], ["i0_dv2_delta_5", 8772.409015655518], ["i0_v3_delta_20", 8637.194492340088], ["i0_v1_delta_20", 8502.04063987732], ["i0_dp1_delta_1", 8461.160507202148], ["a3_delta_1", 8422.078002929688], ["i1_v15_delta_5", 8380.606876373291], ["i0_v11_delta_100", 8271.248374938965], ["i1_dv1_delta_20", 8221.092596054077], ["i1_dp2_delta_5", 8169.577621459961], ["i0_dv3_delta_1", 8166.553550720215], ["i1_p12", 8153.668548583984], ["i0_dv0_delta_5", 8102.739112854004], ["i0_dp3_delta_1", 7949.26993560791], ["i0_p13_delta_1", 7804.123300552368], ["i0_dp2_delta_20", 7779.643707275391], ["i1_dp3", 7671.5197105407715], ["i1_v0_delta_5", 7663.893352508545], ["i1_p9", 7392.292366027832], ["i1_p19", 7369.71798324585], ["i1_dv2_delta_5", 7336.52409362793], ["i0_dp1_delta_5", 7143.295001983643], ["i1_v9_delta_20", 7096.472217559814], ["i0_v8_delta_1", 6979.022693634033], ["i0_v2_delta_20", 6975.824104309082], ["i0_v1_delta_5", 6969.009029388428], ["i0_p5_delta_1", 6824.875194549561], ["i0_dp0_delta_100", 6796.222808837891], ["i0_v12_delta_1", 6415.351345062256], ["i1_v3_delta_100", 6353.574819564819], ["i0_p1_delta_1", 6288.405950546265], ["i0_v2_delta_5", 6200.886787414551], ["a2_delta_5", 6181.944194793701], ["i0_dp2_delta_100", 6142.426719665527], ["i0_v10_delta_20", 6131.606609344482], ["i1_p2", 6107.71967124939], ["i0_v16_delta_20", 6085.296907424927], ["i1_v19_delta_20", 6077.749961853027], ["i0_v14_delta_1", 6070.530490875244], ["i1_p5_delta_5", 6018.330280303955], ["i1_dp2_delta_1", 6016.8440589904785], ["i0_v6_delta_1", 5996.816900253296], ["a3_delta_100", 5905.648994445801], ["i1_v2_delta_100", 5899.248704910278], ["i1_dp2_delta_20", 5873.063312530518], ["i0_dp1_delta_20", 5866.620487213135], ["i1_p3", 5723.137500762939], ["i1_p14", 5653.865001678467], ["i1_v20_delta_5", 5628.691293716431], ["i0_v14_delta_100", 5611.081504821777], ["i1_p18_delta_1", 5607.602840423584], ["i1_v15_delta_100", 5593.677579879761], ["i1_v0_delta_100", 5576.711711883545], ["i1_dv2_delta_20", 5549.28279876709], ["i1_v0_delta_20", 5519.764797210693], ["i0_p17_delta_5", 5514.710506439209], ["i1_v5_delta_100", 5509.49649810791], ["i0_v11_delta_5", 5459.145690917969], ["i0_p6_delta_1", 5426.7822265625], ["i0_v7_delta_20", 5406.6989097595215], ["a5_delta_20", 5350.434823989868], ["i0_v12_delta_5", 5341.096977233887], ["i0_p2_delta_1", 5318.92733001709], ["i1_dv3_delta_1", 5093.201316833496], ["i1_dp0_delta_20", 5068.6873207092285], ["i0_v11_delta_20", 5046.148212432861], ["i1_dv3_delta_5", 5007.255207061768], ["i0_p20_delta_1", 4986.170513153076], ["i1_v13_delta_5", 4947.692985534668], ["i1_v11_delta_100", 4946.55632019043], ["i1_dp1_delta_20", 4946.362400054932], ["i0_v10_delta_100", 4933.677091598511], ["i0_p21_delta_1", 4912.546237945557], ["i0_p12_delta_1", 4845.711902618408], ["i0_v12_delta_100", 4737.475696563721], ["a2_delta_100", 4657.486713409424], ["i1_p18", 4656.042839050293], ["i0_p5_delta_20", 4590.537158966064], ["i1_v18_delta_100", 4578.98420715332], ["i1_dp1_delta_5", 4506.79832649231], ["i0_v8_delta_20", 4506.536994934082], ["i1_v1_delta_100", 4446.605091094971], ["i1_v5_delta_20", 4400.6747970581055], ["i1_v20_delta_20", 4390.629505157471], ["i1_v9_delta_100", 4388.637596130371], ["i1_v10_delta_100", 4327.670387268066], ["i1_v7_delta_100", 4291.731393814087], ["i1_v21_delta_5", 4264.254981994629], ["i1_v21_delta_100", 4263.772602081299], ["i0_v12_delta_20", 4117.99630355835], ["i1_p15_delta_1", 4103.631092071533], ["i1_p11", 4098.501575469971], ["i1_p8_delta_20", 4065.2188091278076], ["i0_p16_delta_20", 4064.936210632324], ["i1_p4", 4043.215888977051], ["i1_v18_delta_20", 4003.175998687744], ["i0_v14_delta_20", 4002.804000854492], ["i1_v13_delta_100", 3983.599784851074], ["i0_p4_delta_1", 3974.1474685668945], ["i1_p3_delta_1", 3936.8714179992676], ["i0_v11_delta_1", 3936.8405990600586], ["a4_delta_5", 3925.2241020202637], ["i1_dp0_delta_1", 3900.9146671295166], ["i1_dv0_delta_1", 3883.4126834869385], ["i1_v19_delta_100", 3871.496894836426], ["i1_p8_delta_100", 3844.7993202209473], ["i1_p0_delta_1", 3833.0067291259766], ["i1_v3_delta_5", 3757.8350048065186], ["i1_v2_delta_20", 3755.9974098205566], ["a2_delta_20", 3752.5897998809814], ["i0_p16_delta_5", 3664.1686935424805], ["i1_v20_delta_100", 3647.749698638916], ["i0_v7_delta_5", 3614.325786590576], ["i0_v17_delta_1", 3612.6328983306885], ["i0_p15_delta_1", 3572.8265228271484], ["i1_dv0_delta_5", 3567.136989593506], ["i1_p11_delta_1", 3533.203441619873], ["i0_p5_delta_100", 3509.2433853149414], ["i0_p20_delta_5", 3462.2647132873535], ["i0_p0_delta_1", 3461.996269226074], ["i0_p3_delta_1", 3457.455307006836], ["i0_p12_delta_5", 3425.9477310180664], ["i1_p14_delta_20", 3412.1541175842285], ["i1_v17_delta_100", 3412.1101989746094], ["i0_p5_delta_5", 3325.0784187316895], ["i1_p7", 3237.8275146484375], ["i0_dp3_delta_5", 3220.9751682281494], ["i0_dv2_delta_20", 3211.426315307617], ["a7_delta_1", 3197.5942001342773], ["i1_v6_delta_20", 3180.4255142211914], ["i1_v17_delta_20", 3176.056390762329], ["i1_p10_delta_100", 3118.0778312683105], ["i1_v21_delta_20", 3089.5779991149902], ["i1_p20", 3089.4174194335938], ["i1_v16_delta_5", 3088.8614959716797], ["i0_p17_delta_1", 3062.2572174072266], ["i0_v10_delta_1", 3061.3153190612793], ["i0_dp1_delta_100", 3058.338520050049], ["a5_delta_100", 3043.1671085357666], ["i1_v6_delta_5", 3026.486204147339], ["i1_v14_delta_20", 3002.5017280578613], ["i1_p8_delta_1", 2987.6333770751953], ["i1_v3_delta_1", 2984.0823249816895], ["i1_p6", 2983.7937927246094], ["i1_v12_delta_100", 2981.799814224243], ["a1", 2981.0748023986816], ["i1_v3_delta_20", 2968.619094848633], ["i0_p11_delta_1", 2966.5469913482666], ["i0_p13_delta_5", 2935.4919929504395], ["i0_v7_delta_1", 2933.4108123779297], ["i1_v6_delta_100", 2913.356792449951], ["i0_p11_delta_5", 2900.715585708618], ["i1_p14_delta_100", 2881.9329986572266], ["i0_p19_delta_20", 2853.0590896606445], ["i1_p0_delta_100", 2852.096878051758], ["i1_v13_delta_20", 2851.071201324463], ["i1_p19_delta_1", 2837.701519012451], ["i1_v7_delta_20", 2825.4173069000244], ["i1_v16_delta_100", 2816.309476852417], ["i1_p5_delta_20", 2812.4047050476074], ["i1_p21_delta_20", 2810.8137741088867], ["i0_p10_delta_1", 2774.5908164978027], ["i0_p8_delta_1", 2753.2766647338867], ["a4_delta_20", 2694.346393585205], ["a4_delta_1", 2642.4539222717285], ["i1_p21_delta_1", 2639.2978858947754], ["i1_v14_delta_100", 2618.869987487793], ["i1_p1", 2615.8245811462402], ["a7_delta_20", 2555.205924987793], ["i1_p8_delta_5", 2531.7120037078857], ["i1_dv3_delta_20", 2528.5123977661133], ["i0_dv3_delta_20", 2510.9423027038574], ["i1_v5_delta_5", 2499.9076042175293], ["i1_v11_delta_20", 2491.575309753418], ["i1_v2_delta_5", 2485.992118835449], ["i0_p9_delta_1", 2481.4198875427246], ["i0_p11_delta_20", 2478.6981887817383], ["i1_v8_delta_20", 2470.129707336426], ["i1_v9_delta_1", 2460.1190032958984], ["i0_dv2_delta_100", 2441.4424209594727], ["i0_p17_delta_20", 2428.8712005615234], ["i0_p8_delta_5", 2428.640811920166], ["i1_v8_delta_100", 2425.3673095703125], ["i1_p5_delta_1", 2392.6044921875], ["i1_p8", 2388.1704025268555], ["a1_delta_5", 2383.413703918457], ["i1_p7_delta_20", 2370.0730934143066], ["i1_v17_delta_5", 2341.114200592041], ["i0_dv0_delta_20", 2329.1811180114746], ["i1_v11_delta_5", 2313.4304008483887], ["a0_delta_100", 2293.903709411621], ["i1_v8_delta_5", 2258.5963973999023], ["i1_p7_delta_100", 2251.772602081299], ["i1_p17_delta_5", 2248.932186126709], ["i1_p1_delta_1", 2245.588954925537], ["i1_v10_delta_20", 2233.7562160491943], ["i1_dv0_delta_20", 2213.7269020080566], ["i0_p7_delta_1", 2199.4636611938477], ["i1_p0", 2192.9404945373535], ["i1_v7_delta_5", 2164.099298477173], ["i0_p13_delta_20", 2163.662311553955], ["a2_delta_1", 2161.98380279541], ["i1_p15", 2161.9226303100586], ["i1_v12_delta_20", 2143.1041870117188], ["a0_delta_5", 2140.719596862793], ["i0_p9_delta_20", 2126.393096923828], ["i1_v0_delta_1", 2120.9657821655273], ["a0_delta_20", 2113.7402019500732], ["i1_v16_delta_20", 2112.2973022460938], ["i1_p10", 2096.652313232422], ["i0_p19_delta_5", 2076.478187561035], ["i0_p4_delta_5", 2076.1680068969727], ["i0_p18_delta_1", 2036.361011505127], ["a7_delta_100", 2033.4985008239746], ["i1_v1_delta_5", 2011.4416007995605], ["i0_p1_delta_5", 1959.3117713928223], ["a4_delta_100", 1959.1945190429688], ["a7_delta_5", 1957.613489151001], ["i1_p14_delta_1", 1925.216209411621], ["i1_p21", 1907.2612800598145], ["i0_p3_delta_20", 1905.6047134399414], ["i1_p10_delta_20", 1901.1973972320557], ["i1_v10_delta_5", 1896.7737083435059], ["i1_dp1_delta_1", 1890.2450981140137], ["i1_p0_delta_20", 1888.345573425293], ["i0_p0_delta_20", 1865.781681060791], ["i1_dp0_delta_100", 1845.0733032226562], ["i1_v1_delta_20", 1831.8849983215332], ["i1_p7_delta_1", 1751.1374282836914], ["i1_v18_delta_1", 1746.6502990722656], ["i1_p14_delta_5", 1742.3168907165527], ["a1_delta_1", 1729.367099761963], ["i1_p12_delta_1", 1677.0807075500488], ["a0_delta_1", 1672.008394241333], ["i0_p13_delta_100", 1671.3832817077637], ["i1_p3_delta_20", 1654.607021331787], ["i0_p15_delta_5", 1654.6041145324707], ["i1_v20_delta_1", 1617.4307022094727], ["i1_p12_delta_5", 1617.1424942016602], ["i0_p10_delta_5", 1610.9081916809082], ["i1_p20_delta_20", 1590.1994972229004], ["i0_p16_delta_100", 1573.778305053711], ["i0_p4_delta_100", 1564.8250846862793], ["i1_p15_delta_100", 1557.006498336792], ["i1_dp1_delta_100", 1544.008201599121], ["i0_p0_delta_5", 1526.8854999542236], ["a1_delta_100", 1521.8281002044678], ["i1_dp2_delta_100", 1520.0012874603271], ["i0_p17_delta_100", 1464.3773002624512], ["i1_p11_delta_5", 1458.9962043762207], ["i1_p12_delta_100", 1456.743200302124], ["i1_p21_delta_100", 1454.8370933532715], ["i0_p8_delta_20", 1452.1115112304688], ["i0_p7_delta_20", 1442.3052978515625], ["i1_v21_delta_1", 1415.6869049072266], ["i0_dp3_delta_20", 1403.9686851501465], ["i0_p3_delta_5", 1403.033706665039], ["i1_dv1_delta_100", 1396.941795349121], ["i1_p20_delta_1", 1393.4169960021973], ["i0_dv0_delta_100", 1384.7552909851074], ["i0_p10_delta_20", 1375.115592956543], ["i1_p12_delta_20", 1374.3302917480469], ["i0_p12_delta_20", 1361.373390197754], ["i0_p7_delta_100", 1358.0959968566895], ["i1_dv2_delta_100", 1345.089199066162], ["i0_p2_delta_20", 1344.516689300537], ["i1_v16_delta_1", 1344.2026042938232], ["i0_p3_delta_100", 1331.9541969299316], ["i0_p9_delta_5", 1324.9530067443848], ["i1_p17_delta_20", 1322.2261962890625], ["i0_p20_delta_20", 1316.798095703125], ["i1_p2_delta_1", 1306.7539978027344], ["i1_p1_delta_20", 1297.7464790344238], ["a1_delta_20", 1295.076789855957], ["i0_p18_delta_100", 1289.1577911376953], ["i0_p16_delta_1", 1283.8280067443848], ["i1_p3_delta_5", 1280.4375953674316], ["i1_v6_delta_1", 1275.2518882751465], ["i1_dv3_delta_100", 1270.33150100708], ["i1_p5_delta_100", 1258.155891418457], ["i1_p11_delta_100", 1243.0748176574707], ["i1_dp3_delta_100", 1232.7438945770264], ["i0_p2_delta_100", 1205.2747116088867], ["i0_p15_delta_20", 1202.4698905944824], ["i0_p4_delta_20", 1183.6874160766602], ["i1_v18_delta_5", 1182.3411026000977], ["i1_p0_delta_5", 1181.3145027160645], ["i1_v19_delta_1", 1169.3686027526855], ["i1_dp3_delta_20", 1154.9221954345703], ["i0_p11_delta_100", 1133.825595855713], ["i1_v10_delta_1", 1117.9682922363281], ["i1_p16_delta_1", 1111.272632598877], ["i0_p9_delta_100", 1104.3349990844727], ["i1_p18_delta_5", 1096.471996307373], ["i1_p7_delta_5", 1085.0802955627441], ["i0_p20_delta_100", 1065.690486907959], ["i1_p15_delta_20", 1055.7615013122559], ["i1_p1_delta_100", 1032.645896911621], ["i1_p16_delta_20", 1020.4976043701172], ["i1_p3_delta_100", 1014.0936164855957], ["i1_p4_delta_20", 988.2446022033691], ["i1_v11_delta_1", 979.1569938659668], ["i1_v8_delta_1", 976.8000946044922], ["i1_dp3_delta_5", 936.0289993286133], ["i1_p13_delta_1", 935.9799156188965], ["i1_v2_delta_1", 935.8710060119629], ["i1_v15_delta_1", 931.8450050354004], ["i0_p1_delta_20", 923.7753105163574], ["i0_p14_delta_20", 911.0959892272949], ["i0_dv3_delta_100", 907.568904876709], ["i0_p7_delta_5", 906.6078033447266], ["i0_p14_delta_100", 904.89208984375], ["i1_p2_delta_100", 896.4216976165771], ["i1_p9_delta_20", 887.6474914550781], ["i1_p17_delta_100", 885.1384963989258], ["i1_v14_delta_5", 883.5946960449219], ["i1_dp3_delta_1", 879.3448066711426], ["i1_p18_delta_20", 875.7560958862305], ["i0_p2_delta_5", 870.5117988586426], ["i0_p1_delta_100", 858.1200981140137], ["i0_p15_delta_100", 846.6439971923828], ["i1_p20_delta_100", 842.9847984313965], ["i1_v5_delta_1", 836.8800926208496], ["i0_p14_delta_5", 831.9440040588379], ["i1_p19_delta_20", 828.5597991943359], ["i0_dp3_delta_100", 803.6322021484375], ["i1_p18_delta_100", 800.2623138427734], ["i1_p13_delta_20", 786.3687973022461], ["i0_p8_delta_100", 767.5039978027344], ["i1_p20_delta_5", 758.4205932617188], ["i1_dv0_delta_100", 742.7744941711426], ["i1_p9_delta_100", 732.9096984863281], ["i1_p15_delta_5", 731.3045082092285], ["i1_p19_delta_5", 692.8025970458984], ["i1_v7_delta_1", 692.0890941619873], ["i0_p12_delta_100", 688.837703704834], ["i0_p18_delta_20", 683.5587921142578], ["i0_p18_delta_5", 681.967700958252], ["i0_p6_delta_100", 677.8604965209961], ["i0_p0_delta_100", 672.3515014648438], ["i0_p19_delta_100", 666.3130950927734], ["i1_p16_delta_100", 625.7230224609375], ["i0_p21_delta_20", 624.3069953918457], ["i1_p9_delta_1", 619.4837055206299], ["i1_p13_delta_100", 610.4296913146973], ["i1_v17_delta_1", 590.4508018493652], ["i1_v12_delta_5", 588.3010025024414], ["i1_v1_delta_1", 576.0127944946289], ["i1_p9_delta_5", 572.4463043212891], ["i1_p13_delta_5", 570.9151992797852], ["i1_v12_delta_1", 568.2400016784668], ["i1_p21_delta_5", 554.1453018188477], ["i1_p6_delta_100", 551.4708938598633], ["i0_p6_delta_5", 547.4925994873047], ["i1_p4_delta_1", 537.5020980834961], ["i1_p2_delta_5", 537.017894744873], ["i1_p10_delta_5", 507.37110900878906], ["i1_p4_delta_5", 504.5617980957031], ["i1_p11_delta_20", 479.60999298095703], ["i0_p6_delta_20", 453.16450119018555], ["i1_p10_delta_1", 442.3356018066406], ["i1_p1_delta_5", 442.00370025634766], ["i1_p6_delta_1", 433.6204948425293], ["i1_p2_delta_20", 429.6487121582031], ["i1_p6_delta_20", 396.27460861206055], ["i1_p6_delta_5", 393.1943950653076], ["i1_p17_delta_1", 365.82979583740234], ["i1_p4_delta_100", 359.5845031738281], ["i1_p19_delta_100", 351.9392967224121], ["i1_v14_delta_1", 326.0778045654297], ["i0_p21_delta_100", 304.143497467041], ["i0_p21_delta_5", 255.2989044189453], ["i1_p16_delta_5", 248.76319885253906], ["i0_p10_delta_100", 236.72410202026367]]
json.dump(importancia, open('importancia.json', 'w'))
g = [v for _, v in importancia]
tot = sum(g)
print(f'{len(g)} features | las 150 mejores tienen el {100*sum(g[:150])/tot:.1f} % de la ganancia')
print('top 10:', [k for k, _ in importancia[:10]])


## Prueba de humo (~3 min)

Recorre exactamente el mismo camino que la corrida larga (features de memoria larga, selección de columnas, entrenamiento y evaluación), con 8 secuencias y 30 árboles.

In [ ]:
!python train_lgbm_deltas_v1.py --train-parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part1/train_part1.parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part2/train_part2.parquet --valid-parquet /kaggle/input/datasets/joaquinfiorina/connectome-valid/valid.parquet --lags 1 5 20 100 --ema-features --top-features 150 --importance importancia.json --learning-rate 0.05 --num-leaves 63 --min-data-in-leaf 200 --feature-fraction 0.5 --bagging-fraction 0.8 --lambda-l2 1.0 --threads -1 --train-groups 8 --rows-per-group 200 --valid-groups 12 --n-estimators 30 --output-dir /kaggle/tmp/smoke --run-name humo
import os, json
assert os.path.isfile('/kaggle/tmp/smoke/humo/metrics.json'), 'La prueba de humo FALLO'
m = json.load(open('/kaggle/tmp/smoke/humo/metrics.json'))
print('Prueba de humo OK | WP:', round(m['lgbm_global']['weighted_pearson'], 4))
esp = json.load(open('/kaggle/tmp/smoke/humo/feature_spec.json'))
assert esp['ema'] is True, 'no se usaron las features de memoria larga'
assert len(esp['names']) == 190, f"esperaba 190 columnas, hay {len(esp['names'])}"
ema = [n for n in esp['names'] if 'ema' in n or 'vol_lenta' in n]
print(f'columnas: {len(esp["names"])} (de las cuales {len(ema)} son de memoria larga)')


## Los árboles, con todo

10.607 secuencias × 1.000 filas = **10,6M filas**, 190 columnas, 600 árboles por target.

Evaluación en **200 secuencias** de valid (septiembre usó 10, demasiado pocas para distinguir 0.002).

In [ ]:
!python train_lgbm_deltas_v1.py --train-parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part1/train_part1.parquet /kaggle/input/datasets/joaquinfiorina/connectome-train-part2/train_part2.parquet --valid-parquet /kaggle/input/datasets/joaquinfiorina/connectome-valid/valid.parquet --lags 1 5 20 100 --ema-features --top-features 150 --importance importancia.json --learning-rate 0.05 --num-leaves 63 --min-data-in-leaf 200 --feature-fraction 0.5 --bagging-fraction 0.8 --lambda-l2 1.0 --threads -1 --train-groups all --rows-per-group 1000 --valid-groups 200 --n-estimators 600 --seed 2026 --output-dir /kaggle/working/results --run-name arboles_v2 || echo 'FALLO arboles_v2'


## Resumen

In [ ]:
import json, os
d = '/kaggle/working/results/arboles_v2'
m = json.load(open(d + '/metrics.json')) if os.path.isfile(d + '/metrics.json') else None
if m:
    s = m['lgbm_global']['weighted_pearson']
    print(f"ARBOLES v2: {s:.4f}   (t0 {m['lgbm_global']['t0']:.4f}, t1 {m['lgbm_global']['t1']:.4f})")
    print(f'septiembre:  0.6253')
    print(f'diferencia:  {s - 0.6253:+.4f}')
    print()
    print('umbral para aportar al ensamble: 0.6755 x 0.918 = 0.6200')
    print('ensamble de 4 redes en estas mismas 200 secuencias: 0.6755')
    print()
    if s >= 0.650:
        print('VEREDICTO: SIRVE. Ganancia esperada en el ensamble: +0.004 o mas')
    elif s >= 0.635:
        print('VEREDICTO: aporta poco pero real (+0.001 a +0.002). Hay que ver si paga el tiempo')
    elif s > 0.6253:
        print('VEREDICTO: mejoro pero no alcanza')
    else:
        print('VEREDICTO: los arboles quedan cerrados con evidencia')
    imp = json.load(open(d + '/feature_importance.json'))
    tot = sum(v for _, v in imp)
    ema = sum(v for k, v in imp if 'ema' in k or 'vol_lenta' in k)
    print()
    print(f'las 40 features de memoria larga se llevaron el {100*ema/tot:.1f} % de la ganancia')
    print('top 15:', [k for k, _ in imp[:15]])
    print()
    print(sorted(os.listdir(d)))
else:
    print('FALLO: revisar el error de arriba')
